# MaaS Usage Dashboard (Per-Subscription Token Consumption)

This notebook deploys a Grafana dashboard that tracks **per-subscription token consumption** through the MaaS Gateway.

| What | How |
|------|-----|
| **Token consumption** | Limitador `authorized_hits` metric — tokens consumed per subscription |
| **Request rate** | Limitador `authorized_calls` — successful requests within rate limit |
| **Rate limiting** | Limitador `limited_calls` — rejected requests (HTTP 429) |
| **Auth evaluations** | Authorino `auth_server_evaluations_total` — auth success/failure |

### Architecture

```
  Client Request
       │
       ▼
  MaaS Gateway (Envoy)
       │
       ├──► Authorino (AuthPolicy)      → auth_server_evaluations_total
       │      validates API key
       │      injects X-MaaS-Subscription
       │
       ├──► Limitador (TokenRateLimit)   → authorized_calls, authorized_hits, limited_calls
       │      checks token quota
       │      reports token usage
       │
       ▼
  vLLM (Model Serving)
       │
       ▼
  Prometheus  ──►  Thanos Querier  ──►  Grafana Dashboard
```

### Prerequisites

- Completed `1_observability_setup.ipynb` (Grafana deployed in `monitoring` namespace)
- MaaS enabled with subscriptions (`../2_maas/2_enable_maas.ipynb`)
- `.env` file configured with `CLUSTER_DOMAIN`, `MAAS_API_KEY`

## 1. Prerequisites Check

In [ ]:
import subprocess, json, os, urllib.request, urllib.parse, ssl
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/demo/qwen36-27b")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OCP_TOKEN = token_result.stdout.strip()

THANOS_HOST = f"https://thanos-querier-openshift-monitoring.{CLUSTER_DOMAIN}"
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print(f"Cluster Domain:  {CLUSTER_DOMAIN}")
print(f"Thanos Querier:  {THANOS_HOST}")
print(f"MaaS API Key:    {'configured' if MAAS_API_KEY else 'NOT SET'}")
print(f"OCP Token:       {'configured' if OCP_TOKEN else 'NOT SET (run oc login)'}")

In [ ]:
%%bash
echo "=== Grafana ==="
oc get pods -n monitoring -l app=grafana --no-headers 2>/dev/null || echo "  Not found — run 1_observability_setup.ipynb first"

echo ""
echo "=== Limitador ==="
oc get pods -n kuadrant-system -l app=limitador --no-headers 2>/dev/null || echo "  Not found"

echo ""
echo "=== Authorino ==="
oc get pods -n kuadrant-system -l app=authorino --no-headers 2>/dev/null | head -1 || echo "  Not found"

echo ""
echo "=== MaaS Subscriptions ==="
oc get maassubscriptions -n models-as-a-service --no-headers 2>/dev/null || echo "  Not found"

## 2. Enable Limitador Metrics Collection

By default, the Kuadrant operator only creates a ServiceMonitor for the **operator** pod, not the **Limitador** pod itself. The cell below creates a ServiceMonitor that scrapes `authorized_hits`, `authorized_calls`, and `limited_calls` from the Limitador pod.

| Metric | Source | What It Tracks | Label for Grouping |
|--------|--------|----------------|--------------------|
| `authorized_calls` | Limitador | Successful requests within rate limit | `limitador_namespace` |
| `authorized_hits` | Limitador | Tokens consumed (authorized token count) | `limitador_namespace` |
| `limited_calls` | Limitador | Rejected requests — rate limit exceeded (HTTP 429) | `limitador_namespace` |
| `limitador_up` | Limitador | Health check (1 = healthy) | `instance` |
| `auth_server_evaluations_total` | Authorino | Auth evaluation count (success/failure) | `authconfig` |

### Per-User Correlation

Limitador metrics are grouped by **subscription** (`limitador_namespace`), not by individual user.
To correlate subscription → user → API key:

```
Grafana (subscription-level)  →  MaaS API (key-level)  →  User identity
          │                              │                       │
   authorized_hits             /maas-api/v1/api-keys      username, groups
   by limitador_namespace        search by subscription
```

> **Tip:** For 1:1 subscription-to-user mappings, the dashboard directly shows per-user usage.

In [ ]:
%%bash
echo "=== Creating ServiceMonitor for Limitador pod ==="
oc apply -f manifests/10a-servicemonitor-limitador.yaml

echo ""
echo "=== Verify Limitador metrics endpoint ==="
POD=$(oc get pods -n kuadrant-system -l app=limitador -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
if [ -n "$POD" ]; then
  METRICS=$(oc exec -n kuadrant-system $POD -- curl -s http://localhost:8080/metrics 2>/dev/null)
  echo "$METRICS" | grep -E "^(authorized_|limited_|limitador_up)" | head -10
  if echo "$METRICS" | grep -q "limitador_up 1"; then
    echo ""
    echo "✅ Limitador is healthy and exposing metrics"
  fi
else
  echo "⚠️  Limitador pod not found in kuadrant-system"
fi

echo ""
echo "Note: Wait ~30s for Prometheus to scrape the new target before querying Thanos."

## 3. MaaS Metering Architecture

MaaS tracks token consumption through the **Limitador** rate limiter. Every request that passes through the MaaS Gateway generates Prometheus metrics:

| Metric | Source | What It Tracks | Label for Grouping |
|--------|--------|----------------|---------------------|
| `authorized_calls` | Limitador | Successful requests within rate limit | `limitador_namespace` |
| `authorized_hits` | Limitador | Tokens consumed (authorized token count) | `limitador_namespace` |
| `limited_calls` | Limitador | Rejected requests — rate limit exceeded (HTTP 429) | `limitador_namespace` |
| `limitador_up` | Limitador | Health check (1 = healthy) | `instance` |
| `auth_server_evaluations_total` | Authorino | Auth evaluation count (success/failure) | `authconfig` |

### Per-User Correlation

Limitador metrics are grouped by **subscription** (`limitador_namespace`), not by individual user.
To correlate subscription → user → API key:

```
Grafana (subscription-level)  →  MaaS API (key-level)  →  User identity
          │                              │                       │
   authorized_hits             /maas-api/v1/api-keys      username, groups
   by limitador_namespace        search by subscription
```

> **Tip:** For 1:1 subscription-to-user mappings, the dashboard directly shows per-user usage.

## 4. Query Per-Subscription Token Usage

In [ ]:
queries = {
    "Token Consumption (by subscription)": "sum(authorized_hits) by (limitador_namespace)",
    "Authorized Requests (by subscription)": "sum(authorized_calls) by (limitador_namespace)",
    "Rate-Limited Requests (by subscription)": "sum(limited_calls) by (limitador_namespace)",
    "Authorized Tokens (total)": "sum(authorized_hits)",
    "Authorized Requests (total)": "sum(authorized_calls)",
    "Rate-Limited Requests (total)": "sum(limited_calls)",
    "Limitador Health": "limitador_up",
}

print("MaaS Token Usage (Limitador Prometheus Metrics)")
print("=" * 65)

for label, query in queries.items():
    try:
        url = f"{THANOS_HOST}/api/v1/query?query={urllib.parse.quote(query)}"
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {OCP_TOKEN}"})
        with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
            data = json.loads(resp.read())
            results = data.get("data", {}).get("result", [])
            if results:
                for r in results:
                    ns = r.get("metric", {}).get("limitador_namespace", "")
                    val = r.get("value", ["", "N/A"])[1]
                    suffix = f"  [{ns}]" if ns else ""
                    print(f"  {label:<45} {float(val):>12,.0f}{suffix}")
            else:
                print(f"  {label:<45} {'(no data)':>12}")
    except Exception as e:
        print(f"  {label:<45} Error: {str(e)[:40]}")

print("")
print("Metrics are per-subscription (limitador_namespace).")
print("For per-user breakdown, query MaaS API: /maas-api/v1/api-keys/search")

## 5. Deploy MaaS Usage Grafana Dashboard

In [ ]:
%%bash
echo "=== 1. Creating MaaS Usage Dashboard ConfigMap ==="

oc create configmap grafana-dashboard-maas \
  --from-file=maas-usage.json=manifests/10-dashboard-maas.json \
  -n monitoring --dry-run=client -o yaml | oc apply -f -

echo ""
echo "=== 2. Adding MaaS folder to Grafana dashboard provisioning ==="

oc get configmap grafana-dashboards-config -n monitoring -o jsonpath='{.data.dashboards\.yaml}' | grep -q "maas"
if [ $? -ne 0 ]; then
  cat <<'EOF' | oc apply -f -
apiVersion: v1
kind: ConfigMap
metadata:
  name: grafana-dashboards-config
  namespace: monitoring
data:
  dashboards.yaml: |
    apiVersion: 1
    providers:
      - name: app
        folder: Application
        type: file
        options:
          path: /var/lib/grafana/dashboards/app
      - name: gpu
        folder: GPU
        type: file
        options:
          path: /var/lib/grafana/dashboards/gpu
      - name: llm
        folder: LLM
        type: file
        options:
          path: /var/lib/grafana/dashboards/llm
      - name: maas
        folder: MaaS
        type: file
        options:
          path: /var/lib/grafana/dashboards/maas
EOF
  echo "✅ MaaS folder added to provisioning config"
else
  echo "✅ MaaS folder already in provisioning config"
fi

echo ""
echo "=== 3. Updating Grafana Deployment (add dashboard volume) ==="

HAS_MAAS=$(oc get deployment grafana -n monitoring -o jsonpath='{.spec.template.spec.volumes[*].name}' | tr ' ' '\n' | grep dashboard-maas)
if [ -z "$HAS_MAAS" ]; then
  oc patch deployment grafana -n monitoring --type=json -p '[
    {
      "op": "add",
      "path": "/spec/template/spec/volumes/-",
      "value": {
        "name": "dashboard-maas",
        "configMap": { "name": "grafana-dashboard-maas" }
      }
    },
    {
      "op": "add",
      "path": "/spec/template/spec/containers/0/volumeMounts/-",
      "value": {
        "name": "dashboard-maas",
        "mountPath": "/var/lib/grafana/dashboards/maas"
      }
    }
  ]'
  echo "✅ Grafana deployment patched — pod will restart automatically"
else
  echo "✅ Volume mount already exists. Restarting Grafana..."
  oc rollout restart deployment/grafana -n monitoring
fi

echo ""
echo "=== 4. Waiting for Grafana pod ==="
oc rollout status deployment/grafana -n monitoring --timeout=90s

echo ""
echo "✅ MaaS Usage dashboard deployed"

## 6. Verify Dashboard

In [ ]:
import subprocess, json, os, time
from dotenv import load_dotenv

load_dotenv("../.env")

MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")

if not MAAS_API_KEY or not MODEL_ENDPOINT:
    print("⚠️  MAAS_API_KEY or MODEL_ENDPOINT not set — skipping traffic generation")
else:
    print("Sending 3 test requests to generate metric data...")
    for i in range(3):
        r = subprocess.run(
            ["curl", "-sk", "-X", "POST",
             f"{MODEL_ENDPOINT}/v1/chat/completions",
             "-H", f"Authorization: Bearer {MAAS_API_KEY}",
             "-H", "Content-Type: application/json",
             "-d", json.dumps({
                 "model": MODEL_NAME,
                 "messages": [{"role": "user", "content": "Say hello."}],
                 "max_tokens": 10
             })],
            capture_output=True, text=True, timeout=30
        )
        try:
            resp = json.loads(r.stdout)
            tokens = resp.get("usage", {})
            print(f"  Request {i+1}: HTTP 200 | prompt={tokens.get('prompt_tokens', '?')} completion={tokens.get('completion_tokens', '?')} total={tokens.get('total_tokens', '?')}")
        except json.JSONDecodeError:
            print(f"  Request {i+1}: {r.stdout[:80]}")
        time.sleep(1)

    print("\nWaiting 15s for Prometheus scrape...")
    time.sleep(15)

In [ ]:
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OCP_TOKEN = token_result.stdout.strip()
THANOS_HOST = f"https://thanos-querier-openshift-monitoring.{CLUSTER_DOMAIN}"

import ssl
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print("Verifying Limitador metrics in Prometheus...")
print("=" * 55)

check_queries = [
    ("Token Consumption (total)", "sum(authorized_hits)"),
    ("Authorized Requests (total)", "sum(authorized_calls)"),
    ("Rate-Limited (total)", "sum(limited_calls)"),
]

all_ok = True
for label, query in check_queries:
    try:
        url = f"{THANOS_HOST}/api/v1/query?query={urllib.parse.quote(query)}"
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {OCP_TOKEN}"})
        with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
            data = json.loads(resp.read())
            results = data.get("data", {}).get("result", [])
            if results:
                val = results[0].get("value", ["", "0"])[1]
                print(f"  ✅ {label:<35} {float(val):>10,.0f}")
            else:
                print(f"  ⚠️  {label:<35} (no data yet)")
                all_ok = False
    except Exception as e:
        print(f"  ❌ {label:<35} {str(e)[:40]}")
        all_ok = False

grafana_route = subprocess.run(
    ["oc", "get", "route", "grafana", "-n", "monitoring",
     "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

print("")
print("=" * 55)
if grafana_route:
    print(f"Grafana MaaS Dashboard:")
    print(f"  https://{grafana_route}/d/maas-usage-metrics/")
    print(f"")
    print(f"All Dashboards:")
    print(f"  https://{grafana_route}/dashboards")
else:
    print("⚠️  Grafana route not found")

if all_ok:
    print("\n✅ Metrics are flowing. Open the Grafana dashboard to view panels.")
else:
    print("\n⚠️  Some metrics have no data. Send a few requests through the MaaS Gateway and wait for Prometheus to scrape.")

## 7. Summary

The MaaS Usage dashboard is now deployed in Grafana:

| Component | Namespace | Purpose |
|-----------|-----------|----------|
| Limitador | `kuadrant-system` | Rate limiter — exposes `authorized_hits/calls`, `limited_calls` |
| Authorino | `kuadrant-system` | Auth evaluator — exposes `auth_server_evaluations_total` |
| Grafana Dashboard | `monitoring` | Visualizes per-subscription token consumption |

### Dashboard Panels

| Panel | Metric | What It Shows |
|-------|--------|---------------|
| Token Consumption by Subscription | `authorized_hits` | Token usage rate (tok/min) per subscription |
| Request Rate by Subscription | `authorized_calls` | Successful request rate per subscription |
| Rate-Limited Requests | `limited_calls` | HTTP 429 events per subscription |
| Auth Evaluations | `auth_server_evaluations_total` | Auth success/failure by policy |
| Total Tokens / Requests / 429s (1h) | `increase(...[1h])` | Aggregate counters over last hour |
| Token Usage Breakdown | `authorized_hits` | Horizontal bar gauge comparing subscriptions |
| Limitador Health | `limitador_up` | Rate limiter liveness |

### Per-User Tracking

The dashboard shows usage at the **subscription level**. For per-user detail:

1. **Dashboard** — identify which subscription consumed the most tokens
2. **MaaS API** — query `/maas-api/v1/api-keys/search` to list keys bound to that subscription
3. **API Key metadata** — each key has `username`, `keyId`, `lastUsed` fields

### Related Notebooks

- `1_observability_setup.ipynb` — Deploys Grafana, Prometheus monitors, Tempo tracing
- `../4_control/1_maas_advanced.ipynb` — MaaS subscription configuration and raw PromQL queries
- `../4_control/2_maas_policy_test.ipynb` — Rate limit enforcement testing